# Gold Layer — Denormalized Sales Analytics Tables

This notebook reads the 3 silver CRM tables (`silver_crm_cust_info`, `silver_crm_prd_info`, `silver_crm_sales_details`) and enriches them with ERP source data (`bronze_erp_cust_az12`, `bronze_erp_loc_a101`, `bronze_erp_px_cat_g1v2`) to build flat, reporting-ready tables in the `sales_datalakehouse.gold` schema.

**Gold tables produced:**

1. `gold_customer_enriched` — customer dimension with birth date, gender, and country
2. `gold_sales_denormalized` — sales/product fact with product master info, category, and subcategory

## Step 1 — Read Silver Layer Tables

Load the 3 cleaned silver tables that will serve as the base for the gold layer denormalization.

In [0]:
# Read the 3 silver tables
df_silver_cust = spark.read.table("sales_datalakehouse.silver.silver_crm_cust_info")
df_silver_prd = spark.read.table("sales_datalakehouse.silver.silver_crm_prd_info")
df_silver_sls = spark.read.table("sales_datalakehouse.silver.silver_crm_sales_details")

print("=== Silver Table Row Counts ===")
print(f"silver_crm_cust_info:      {df_silver_cust.count()} rows")
print(f"silver_crm_prd_info:       {df_silver_prd.count()} rows")
print(f"silver_crm_sales_details:  {df_silver_sls.count()} rows")

print("\n=== Silver Customer Schema ===")
df_silver_cust.printSchema()

print("\n=== Silver Product Schema ===")
df_silver_prd.printSchema()

print("\n=== Silver Sales Details Schema ===")
df_silver_sls.printSchema()

## Step 2 — Read & Clean ERP Bronze Tables

The ERP source provides additional attributes not available in CRM:

* `bronze_erp_cust_az12` — customer birth date and gender (joins to `customer_key` via `CONCAT('NASA', SUBSTRING(customer_key, 2))`)
* `bronze_erp_loc_a101` — customer country (joins to `customer_key` via `CONCAT('AW-', SUBSTRING(customer_key, 3))`)
* `bronze_erp_px_cat_g1v2` — product category, subcategory, maintenance flag (joins to `product_key` via first two hyphen-delimited segments joined by underscore)

In [0]:
from pyspark.sql.functions import trim, col, concat, substring, split_part, lit

# Read ERP bronze tables and clean them
df_erp_cust = spark.read.table("sales_datalakehouse.bronze.bronze_erp_cust_az12")
df_erp_loc = spark.read.table("sales_datalakehouse.bronze.bronze_erp_loc_a101")
df_erp_cat = spark.read.table("sales_datalakehouse.bronze.bronze_erp_px_cat_g1v2")

# Clean ERP customer table: trim whitespace, rename columns
df_erp_cust_clean = df_erp_cust.select(
    trim(col("CID")).alias("erp_cust_cid"),
    col("BDATE").alias("birth_date"),
    trim(col("GEN")).alias("erp_gender")
)

# Clean ERP location table: trim whitespace, rename columns
df_erp_loc_clean = df_erp_loc.select(
    trim(col("CID")).alias("erp_loc_cid"),
    trim(col("CNTRY")).alias("country")
)

# Clean ERP product category table: trim whitespace, rename columns
df_erp_cat_clean = df_erp_cat.select(
    trim(col("ID")).alias("category_id"),
    trim(col("CAT")).alias("category"),
    trim(col("SUBCAT")).alias("subcategory"),
    trim(col("MAINTENANCE")).alias("maintenance_required")
)

print("=== ERP Table Row Counts (after cleaning) ===")
print(f"erp_cust_az12:  {df_erp_cust_clean.count()} rows")
print(f"erp_loc_a101:   {df_erp_loc_clean.count()} rows")
print(f"erp_px_cat:     {df_erp_cat_clean.count()} rows")

print("\n=== Sample: ERP Customer ===")
df_erp_cust_clean.show(5)

print("\n=== Sample: ERP Location ===")
df_erp_loc_clean.show(5)

print("\n=== Sample: ERP Product Category ===")
df_erp_cat_clean.show(5)
print("Step 2 complete: ERP tables read and cleaned.")

## Step 3 — Build Denormalized Customer Dimension

Join `silver_crm_cust_info` with ERP customer data (birth date, gender) and ERP location data (country) to create a flat, enriched customer table for reporting.

In [0]:
from pyspark.sql.functions import trim, col, concat, substring, when, lit

# Start with silver customer table and add join keys for ERP
df_gold_cust = df_silver_cust \
    .withColumn("erp_cust_cid", concat(lit("NASA"), substring(col("customer_key"), 2, 100))) \
    .withColumn("erp_loc_cid", concat(lit("AW-"), substring(col("customer_key"), 3, 100)))

# LEFT JOIN with ERP customer data (birth date, gender)
df_gold_cust = df_gold_cust.join(
    df_erp_cust_clean,
    on="erp_cust_cid",
    how="left"
)

# LEFT JOIN with ERP location data (country)
df_gold_cust = df_gold_cust.join(
    df_erp_loc_clean,
    on="erp_loc_cid",
    how="left"
)

# Standardize country values: map variants to canonical names
df_gold_cust = df_gold_cust.withColumn(
    "country",
    when(col("country").isin("USA", "US"), lit("United States"))
    .when(col("country") == "DE", lit("Germany"))
    .when(col("country").isNull(), lit("Unknown"))
    .when(trim(col("country")) == "", lit("Unknown"))
    .otherwise(trim(col("country")))
)

# Standardize ERP gender: M → Male, F → Female, empty → NULL
df_gold_cust = df_gold_cust.withColumn(
    "erp_gender_clean",
    when(col("erp_gender").isin("M"), lit("Male"))
    .when(col("erp_gender").isin("F"), lit("Female"))
    .when(col("erp_gender").isNull(), lit(None))
    .when(trim(col("erp_gender")) == "", lit(None))
    .otherwise(trim(col("erp_gender")))
)

# Enrich gender: when CRM gender is "Unknown", fall back to ERP gender
df_gold_cust = df_gold_cust.withColumn(
    "gender",
    when((col("gender") == "Unknown") & col("erp_gender_clean").isNotNull(), col("erp_gender_clean"))
    .otherwise(col("gender"))
).drop("erp_gender", "erp_gender_clean")

# Select and order columns for the final denormalized customer table
df_gold_cust = df_gold_cust.select(
    col("customer_id"),
    col("customer_key"),
    col("first_name"),
    col("last_name"),
    col("marital_status"),
    col("gender"),
    col("birth_date"),
    col("country"),
    col("created_date")
)

print(f"Denormalized customer row count: {df_gold_cust.count()}")
print("\n=== Gold Customer Schema ===")
df_gold_cust.printSchema()

print("\n=== Sample Data ===")
display(df_gold_cust.limit(10))

# Verify ERP enrichment coverage
total = df_gold_cust.count()
with_country = df_gold_cust.where(col("country").isNotNull()).count()
with_birthdate = df_gold_cust.where(col("birth_date").isNotNull()).count()
print(f"\n=== ERP Enrichment Coverage ===")
print(f"Rows with country:    {with_country}/{total} ({round(with_country/total*100, 1)}%)")
print(f"Rows with birth_date: {with_birthdate}/{total} ({round(with_birthdate/total*100, 1)}%)")
print("Step 3 complete: Denormalized customer dimension built.")

## Step 4 — Build Denormalized Sales/Product Table

Join `silver_crm_sales_details` with `silver_crm_prd_info` (product master) and `bronze_erp_px_cat_g1v2` (product category/subcategory) to create a flat, enriched sales table for reporting.

In [0]:
from pyspark.sql.functions import trim, col, concat, split, element_at, when, lit

# Start with silver sales_details as the base (fact table)
df_gold_sls = df_silver_sls.alias("sls")

# LEFT JOIN with silver prd_info (product master) on product_id
df_gold_sls = df_gold_sls.join(
    df_silver_prd.alias("prd"),
    on="product_id",
    how="left"
)

# Add derived category_id for ERP product category join
# Use split() + element_at() instead of split_part() for Spark Connect compatibility
df_gold_sls = df_gold_sls.withColumn(
    "category_id",
    concat(
        element_at(split(col("sls.product_key"), "-"), 1),
        lit("_"),
        element_at(split(col("sls.product_key"), "-"), 2)
    )
)

# LEFT JOIN with ERP product category
df_gold_sls = df_gold_sls.join(
    df_erp_cat_clean,
    on="category_id",
    how="left"
)

# Select and order columns for the final denormalized sales/product table
# Use product info from prd_info (master), with sales_details as the base
df_gold_sls = df_gold_sls.select(
    col("product_id"),
    col("sls.product_key").alias("product_key"),
    col("sls.product_name").alias("product_name"),
    col("sls.cost").alias("cost"),
    col("sls.product_line").alias("product_line"),
    col("sls.start_date").alias("product_start_date"),
    col("sls.end_date").alias("product_end_date"),
    col("category"),
    col("subcategory"),
    col("maintenance_required"),
    # Product master columns for cross-reference
    col("prd.product_name").alias("master_product_name"),
    col("prd.cost").alias("master_cost"),
    col("prd.product_line").alias("master_product_line")
)

print(f"Denormalized sales/product row count: {df_gold_sls.count()}")
print("\n=== Gold Sales Schema ===")
df_gold_sls.printSchema()

print("\n=== Sample Data ===")
display(df_gold_sls.limit(10))

# Verify ERP category enrichment coverage
total = df_gold_sls.count()
with_category = df_gold_sls.where(col("category").isNotNull()).count()
print(f"\n=== ERP Category Enrichment Coverage ===")
print(f"Rows with category: {with_category}/{total} ({round(with_category/total*100, 1)}%)")
print("Step 4 complete: Denormalized sales/product table built.")

## Step 5 — Write Denormalized Tables to Gold Layer

Write both denormalized tables to the `sales_datalakehouse.gold` schema as managed Delta tables, ready for BI dashboards and ad-hoc analytics.

In [0]:
# Create gold schema if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS sales_datalakehouse.gold")

# Write denormalized customer table to gold
gold_cust_table = "sales_datalakehouse.gold.gold_customer_enriched"
df_gold_cust.write.mode("overwrite").saveAsTable(gold_cust_table)
print(f"Saved to gold table: {gold_cust_table}")
print(f"Row count: {spark.read.table(gold_cust_table).count()}")

# Write denormalized sales/product table to gold
gold_sls_table = "sales_datalakehouse.gold.gold_sales_denormalized"
df_gold_sls.write.mode("overwrite").saveAsTable(gold_sls_table)
print(f"\nSaved to gold table: {gold_sls_table}")
print(f"Row count: {spark.read.table(gold_sls_table).count()}")

print("\nStep 5 complete: Both gold tables written.")

## Step 6 — Verify Gold Layer Tables

Final verification: show table schemas, sample data, and summary statistics for both gold tables.

In [0]:
print("=" * 60)
print("GOLD LAYER VERIFICATION")
print("=" * 60)

# --- Verify gold_customer_enriched ---
print("\n1. gold_customer_enriched")
print("-" * 40)
df_verify_cust = spark.read.table("sales_datalakehouse.gold.gold_customer_enriched")
print(f"Row count: {df_verify_cust.count()}")
print("Schema:")
df_verify_cust.printSchema()
print("\nSample data:")
display(df_verify_cust.limit(10))

# Summary stats
print("\nCountry distribution:")
df_verify_cust.groupBy("country").count().orderBy("count", ascending=False).show()

print("\nGender distribution:")
df_verify_cust.groupBy("gender").count().orderBy("count", ascending=False).show()

# --- Verify gold_sales_denormalized ---
print("\n2. gold_sales_denormalized")
print("-" * 40)
df_verify_sls = spark.read.table("sales_datalakehouse.gold.gold_sales_denormalized")
print(f"Row count: {df_verify_sls.count()}")
print("Schema:")
df_verify_sls.printSchema()
print("\nSample data:")
display(df_verify_sls.limit(10))

# Summary stats
print("\nProduct line distribution:")
df_verify_sls.groupBy("product_line").count().orderBy("count", ascending=False).show()

print("\nCategory distribution:")
df_verify_sls.groupBy("category").count().orderBy("count", ascending=False).show()

print("\n" + "=" * 60)
print("Gold layer verification complete!")
print("=" * 60)